# DEC5 000973: native-noise and subpixel TSDF controls

## tl;dr
Neither control passes the native skin/tube gate: native NLM has 0/6 accepted variant-anchor outputs; subpixel TSDF has 0/3. Strong NLM worsens F face LPIPS from .047989 to .130522. A 10.66-million-triangle mesh still has the patch, wrong tube contour and extra hand holes.

## Context & Methods
These are paired experiments at **one temporal frame**, not nine temporal samples. F/J/L held RGB is used only after prediction. Native filtering leaves mesh, visibility, source labels and UVs fixed. Subpixel fusion instead changes voxel/truncation with the same 62 raw depths and original train RGB.

### Key Assumptions
The HH-MAD native-image statistic is a high-frequency floor, not identified sensor noise or a lens PSF. The study face ROI excludes most neck/hand and differs from the original campaign ROI. Native visual review is therefore essential. A truncation-band projection is not an actual geometric-error estimate.

## Data
Artifacts are immutable-input controls under `/mnt/data/lookcloser_dec5_5a3_surface_repair`. Run on clever-shadow with the repository `.venv`, NumPy and Pillow. This notebook verifies existing artifacts; it does not reconstruct, change images or fit to held RGB.


In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
from PIL import Image

root = Path('/mnt/data/lookcloser_dec5_5a3_surface_repair')
def read(path):
    return json.loads(Path(path).read_text())

def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

noise = read(root/'native_noise_control/findings.json')
subpixel = read(root/'subpixel_tsdf_control/findings.json')
for name, findings in [('native_noise', noise), ('subpixel_tsdf', subpixel)]:
    assert findings['audit_status'] == 'comparison_integrity_pass_candidate_quality_fail'
    assert findings['visual_pass'] == 0 and not findings['temporal_promotion']
    for path, expected in findings['retained_hashes'].items():
        assert sha(path) == expected, path
    print(name, 'verified retained hashes:', len(findings['retained_hashes']))
assert noise['temporal_frame_count'] == 1 and noise['variant_anchor_count'] == 6
assert len({(r['anchor'], r['variant']) for r in noise['rows']}) == 6
assert noise['visual_fail'] == 6 and subpixel['visual_fail'] == 3


native_noise verified retained hashes: 309
subpixel_tsdf verified retained hashes: 111


## Results

### Exact disabled replays and native-cache provenance
A fresh reprojection of unfiltered native RGB must reproduce all frozen controls exactly. The same physical camera/image gets the same native filter across target views.


In [2]:
held = {'F004_B005_1210O9', 'J004_D005_1210TA', 'L004_B005_12106A'}
cache_hashes = {}
for anchor in ['F', 'J', 'L']:
    folder = root/'native_noise_control'/anchor
    request = read(folder/'request.json')
    assert request['train_camera_count'] == 62 and request['source_camera_count'] == 8
    assert not held.intersection(request['sources'])
    for key in ['geometry_unchanged', 'visibility_unchanged', 'hard_labels_unchanged', 'texture_uvs_unchanged']:
        assert request[key]
    assert not request['uses_eval_rgb'] and not request['uses_semantic_masks']
    baseline = root/'base_transport_control'/anchor/'control'
    assert sha(folder/'source_selection.png') == sha(baseline/'source_selection.png')
    assert sha(folder/'native_off/eval_pred_0000.png') == sha(baseline/'baseline_pred_0000.png')
    sources = read(folder/'source_filter_audit.json')['sources']
    assert max(s['native_replay_max_valid_rgb8_error'] for s in sources) == 0
    for source in sources:
        physical = source['physical_camera']
        cache_path = Path(source['native_cache_manifest'])
        assert sha(cache_path) == source['native_cache_manifest_sha256']
        cache = read(cache_path)
        assert not cache['high_frequency_floor']['physical_sensor_noise_identified']
        assert physical not in held and cache['physical_camera'] == physical
        assert sha(cache['source']) == cache['source_sha256']
        if physical in cache_hashes:
            assert cache_hashes[physical] == sha(cache_path)
        cache_hashes[physical] = sha(cache_path)
        for name, expected in cache['outputs'].items():
            assert sha(cache_path.parent/name) == expected
    print(anchor, 'byte-identical unfiltered replay; 8 train-source warps match exactly')
assert len(cache_hashes) == noise['unique_native_sources'] == 23
print('Unique native images:', len(cache_hashes))


F byte-identical unfiltered replay; 8 train-source warps match exactly
J byte-identical unfiltered replay; 8 train-source warps match exactly
L byte-identical unfiltered replay; 8 train-source warps match exactly
Unique native images: 23


### Paired face-only metrics
All rows below use the same held F GT, manual study face polygon and metric implementation. They are not comparable with the old campaign's shifted ROI.


In [3]:
metric_keys = ['face_psnr', 'face_ssim', 'face_lpips']
baseline = read(root/'base_transport_control/F/metrics_baseline/metrics.json')
paths = [
    ('native_off', root/'native_noise_control/F/metrics_native_off/metrics.json'),
    ('nlm1', root/'native_noise_control/F/metrics_nlm1/metrics.json'),
    ('nlm2', root/'native_noise_control/F/metrics_nlm2/metrics.json'),
    ('prior_finer_tsdf', root/'full_block_narrow_control/finer_narrow/face_metrics/metrics.json'),
    ('subpixel_tsdf', root/'subpixel_tsdf_control/subpixel/face_metrics/metrics.json'),
]
metrics = {}
for name, path in paths:
    m = read(path)
    for key in ['ground_truth_sha256', 'face_polygons_sha256', 'face_bbox_xyxy', 'protocol', 'metric_domain']:
        assert m[key] == baseline[key]
    for key in ['prediction', 'ground_truth', 'face_polygons']:
        assert sha(m[key]) == m[key+'_sha256']
    assert all(np.isfinite(m[key]) for key in metric_keys)
    assert not any(key.startswith(('loss', 'full_frame', 'room_psnr', 'actor_psnr')) for key in m)
    metrics[name] = m
    print(name, {key: round(m[key], 6) for key in metric_keys})
assert all(metrics['native_off'][key] == baseline[key] for key in metric_keys)
print('NLM2 LPIPS / baseline:', round(metrics['nlm2']['face_lpips']/baseline['face_lpips'], 3))
print('Fixed study face bounding box:', baseline['face_bbox_xyxy'])


native_off {'face_psnr': 28.724285, 'face_ssim': 0.889098, 'face_lpips': 0.047989}
nlm1 {'face_psnr': 28.726891, 'face_ssim': 0.889567, 'face_lpips': 0.049411}
nlm2 {'face_psnr': 28.786858, 'face_ssim': 0.898417, 'face_lpips': 0.130522}
prior_finer_tsdf {'face_psnr': 28.766863, 'face_ssim': 0.89186, 'face_lpips': 0.048132}
subpixel_tsdf {'face_psnr': 28.761492, 'face_ssim': 0.891466, 'face_lpips': 0.047728}
NLM2 LPIPS / baseline: 2.72
Fixed study face bounding box: [700, 388, 1165, 726]


### Subpixel mesh and completed native review
The independent artifact auditor reloaded the mesh and recomputed connected components, checked all 62 raw arrays, finite PNG/EXR pairs, source selections and reprojection audits. Here we reconcile its retained inventory and the actually inspected crop lists.


In [4]:
request = read(root/'subpixel_tsdf_control/request.json')
mesh = read(root/'subpixel_tsdf_control/subpixel/mesh.json')
assert request['variants'] == [['subpixel', .0000625, .0005]]
assert len(request['depth_hashes']) == subpixel['depth_maps_verified'] == 62
assert subpixel['depth_shape'] == [1080, 1920]
assert mesh['vertices'] == 5450524 and mesh['triangles'] == 10657121
assert mesh['component_triangles'] == [10410840, 87186, 39872, 39026, 31499, 24555, 24143]
assert sum(mesh['component_triangles']) == mesh['triangles']
assert mesh['connected_components'] == 7
assert not mesh['parameters']['full_block_inventory']['raw_volume_serialized']
assert sha(root/'subpixel_tsdf_control/subpixel/mesh.ply') == mesh['output_sha256']
print('Mesh vertices / triangles / components:', mesh['vertices'], mesh['triangles'], mesh['connected_components'])
print('Depth coverage mean / min:', subpixel['depth_coverage_mean'], subpixel['depth_coverage_min'])

noise_crops = [path for row in noise['rows'] for path in row['inspected_crops']]
subpixel_review = read(root/'subpixel_tsdf_control/visual_review.json')
subpixel_crops = subpixel_review['inspected_crops']
assert len(noise_crops) == len(set(noise_crops)) == 14
assert len(subpixel_crops) == len(set(subpixel_crops)) == 7
for path in noise_crops + subpixel_crops:
    assert Path(path).is_file()
    review_inputs = read(Path(path).parent/'review_inputs.json')
    assert review_inputs['rotate'] == 90 and len(review_inputs['images']) == 3
    box_spec = next(spec for spec in review_inputs['boxes'] if spec.split(':')[0] == Path(path).stem)
    x0, y0, x1, y1 = map(int, box_spec.split(':')[1].split(','))
    with Image.open(path) as crop:
        assert crop.size == (3 * (y1-y0), x1-x0+28), path
    for image in review_inputs['images']:
        assert sha(image['path']) == image['sha256']
print('Actually inspected saved crops:', len(noise_crops), '+', len(subpixel_crops))
print('Native quality gates: noise 0 pass / 6 fail; subpixel 0 pass / 3 fail.')


Mesh vertices / triangles / components: 5450524 10657121 7
Depth coverage mean / min: 0.38273008917015133 0.24788242669753086


Actually inspected saved crops: 14 + 7
Native quality gates: noise 0 pass / 6 fail; subpixel 0 pass / 3 fail.


## Takeaways
- Mild native filtering barely changes the patch; stronger filtering loses detail without fixing it.
- Smaller voxels produce 10,657,121 triangles but still fail all three anchors and add hand holes. Mesh density is not evidence of correctness.
- Exact-off replays and paired hashes protect this comparison from changed inputs or ROI definitions. They do not prove a unique exposure, BRDF, focus or calibration cause.
- These are retained negative controls, not production changes. No repaired every-40th-available-frame or continuous fly-through recipe has passed.
- Native review images and the full experiment narrative are linked in [the experiment report](../dec5_patchmatch_surface_repair.md).
